# Notebook 3 — Drift Validation & Visualization

**Purpose:** Validate the presence and type of data drift in the three datasets
used in this study, and provide visual evidence supporting the experimental design.

**Research context:** Before running the retraining simulation (Notebook 4),
it is necessary to confirm that: (1) covariate drift exists in the synthetic datasets,
(2) concept drift is present and causes predictive degradation, and (3) the real
Electricity dataset exhibits naturally occurring distributional fluctuations.
This notebook produces the evidence for all three claims.

**Datasets validated (3 datasets):**
| Dataset | Expected Drift Type |
|---------|--------------------|
| Electricity (Real) | Natural covariate + concept drift (irregular) |
| Synthetic Gradual Covariate + Concept Drift | Both, gradual — P(X) and P(Y|X) shift gradually |
| Synthetic Abrupt Covariate + Concept Drift | Both, abrupt — P(X) and P(Y|X) shift as step function |

> **Note:** The covariate-only variants (Gradual Covariate and Abrupt Covariate)
> are excluded from this notebook because they do not exhibit concept drift
> and are not part of the experimental evaluation in Chapter 4.

**Plots produced per dataset (6 × 3 = 18 total):**
1. PSI vs Baseline Reference — confirms presence/magnitude of covariate drift
2. PSI Inter-Batch — shows local distribution stability between consecutive batches
3. Accuracy Without Retraining — confirms concept drift through predictive degradation
4. Combined PSI + Accuracy — identifies benign vs malignant drift episodes
5. Feature Importance Drift — shows which features drive the distributional shift
6. Per-Feature PSI Heatmap — granular view of drift signal per feature over time

**Key parameter:** PSI threshold = 0.25 (significant drift).
Batches with PSI ≥ 0.25 and stable accuracy are classified as *false drift*
(benign covariate shift). Batches with PSI < 0.25 and degraded accuracy are
classified as *missed drift* (concept-only drift without covariate signal).

---
| Cell | Content |
|------|---------|
| 1 | Install, Imports & Configuration (3 dataset definitions) |
| 2 | Utility Functions (load, PSI, model, plot helpers) |
| 3 | PSI Computation Functions |
| 4 | Individual Plot Functions (Plots 1–5) + PSI Heatmap (Plot 6) |
| 5 | Summary Table Builder |
| 6 | Main Execution Loop (3 datasets × 6 plots = 18 plots) |
| 7 | Cross-Dataset Summary Plots & Interpretation |

## Cell 1 — Install Dependencies, Imports & Configuration

All required packages are installed in a single cell.
Path configuration is centralised here — update `DATA_PATH`
if the Google Drive folder structure differs.

In [1]:
# ── CELL 1 — INSTALL & IMPORTS ───────────────────────────────
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn openpyxl -q

from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── PATHS ────────────────────────────────────────────────────
DATA_PATH = (
    "/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/"
)
PLOT_PATH = os.path.join(DATA_PATH, "drift_validation_plots")
os.makedirs(PLOT_PATH, exist_ok=True)

# ── CONSTANTS ────────────────────────────────────────────────
FEATURES       = ['nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer']
TARGET         = 'class'
BATCH_SIZE     = 500
TOTAL_SAMPLES  = 45_000   # 90 batches x 500 rows — matches electricity_clean_v2
INIT_RATIO     = 0.20     # first 20% = baseline
TRAIN_SPLIT    = 0.80
PSI_BINS       = 10
PSI_MOD        = 0.10     # moderate drift threshold
PSI_SIG        = 0.25     # significant drift threshold

# ── THREE DATASETS USED IN THIS THESIS ──────────────────────
# Only electricity_gradual_concept and electricity_abrupt_concept
# are used as synthetic datasets. The covariate-only variants
# (electricity_gradual, electricity_abrupt) are excluded because
# they do not exhibit concept drift and are not part of the
# experimental evaluation in Chapter 4.
DATASET_FILES = {
    'electricity'               : 'electricity_clean.xlsx',
    'electricity_gradual_concept': 'gradual_concept.xlsx',
    'electricity_abrupt_concept' : 'abrupt_concept.xlsx',
}

DATASET_LABELS = {
    'electricity'               : 'Electricity (Real)',
    'electricity_gradual_concept': 'Gradual Covariate + Concept Drift',
    'electricity_abrupt_concept' : 'Abrupt Covariate + Concept Drift',
}

# Palette
C_PSI_REF    = '#4878cf'
C_PSI_IB     = '#6acc65'
C_ACC        = '#d65f5f'
C_THRESH_MOD = '#f5a623'
C_THRESH_SIG = '#e74c3c'
C_BAND       = '#e8f0fb'

print(f"Datasets: {list(DATASET_LABELS.keys())}")
print(f"Plot output path: {PLOT_PATH}")


Mounted at /content/drive
Datasets: ['electricity', 'electricity_gradual_concept', 'electricity_abrupt_concept']
Plot output path: /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/drift_validation_plots


## Cell 2 — Utility Functions

Core helper functions used throughout the notebook:
- `load_dataset()` — loads dataset from Excel, caps to `TOTAL_SAMPLES`, assigns `batch_id`
- `get_baseline_stream()` — splits data into initial baseline (20%) and streaming batches (80%)
- `compute_psi_feature()` / `compute_psi_mean()` — PSI calculation per feature and aggregated
- `train_fixed_model()` — trains a single Logistic Regression on baseline; never retrained
- `evaluate_stream()` — evaluates the fixed model on each streaming batch

**Note on `TOTAL_SAMPLES = 45,000`:** This matches `electricity_clean_v2.xlsx`
exactly after the 312-row trim applied in Notebook 1, ensuring all datasets
produce identical streaming configurations (90 batches × 500 rows).

In [2]:
def load_dataset(dataset_name: str) -> pd.DataFrame:
    """Load a dataset, add batch_id if missing, cap to TOTAL_SAMPLES."""
    fpath = os.path.join(DATA_PATH, DATASET_FILES[dataset_name])
    df = pd.read_excel(fpath)
    df = df.iloc[:TOTAL_SAMPLES].copy().reset_index(drop=True)
    if 'batch_id' not in df.columns:
        df['batch_id'] = np.arange(len(df)) // BATCH_SIZE
    return df


def get_baseline_stream(df: pd.DataFrame):
    """Split df into baseline pool and list of streaming batch DataFrames."""
    all_batches = sorted(df['batch_id'].unique())
    n_init      = int(len(all_batches) * INIT_RATIO)
    init_ids    = all_batches[:n_init]
    stream_ids  = all_batches[n_init:]
    baseline    = df[df['batch_id'].isin(init_ids)].copy().reset_index(drop=True)
    stream      = [df[df['batch_id'] == b].copy().reset_index(drop=True)
                   for b in stream_ids]
    return baseline, stream, stream_ids


def compute_psi_feature(ref: np.ndarray, cur: np.ndarray,
                         bins: int = PSI_BINS) -> float:
    """PSI for a single feature. Capped at 1.0."""
    breakpoints = np.unique(np.percentile(ref, np.linspace(0, 100, bins + 1)))
    if len(breakpoints) < 2:
        return 0.0
    e = np.histogram(ref, bins=breakpoints)[0] / len(ref)
    a = np.histogram(cur, bins=breakpoints)[0] / len(cur)
    e = np.clip(e, 1e-4, None)
    a = np.clip(a, 1e-4, None)
    return float(min(np.sum((e - a) * np.log(e / a)), 1.0))


def compute_psi_mean(ref_df: pd.DataFrame, cur_df: pd.DataFrame,
                     features: list = FEATURES) -> float:
    """Average PSI across all features."""
    return float(np.mean([
        compute_psi_feature(ref_df[f].values, cur_df[f].values)
        for f in features
    ]))


def compute_psi_per_feature(ref_df: pd.DataFrame, cur_df: pd.DataFrame,
                              features: list = FEATURES) -> dict:
    """PSI per feature dict."""
    return {f: compute_psi_feature(ref_df[f].values, cur_df[f].values)
            for f in features}


def train_fixed_model(baseline: pd.DataFrame) -> LogisticRegression:
    """
    Train ONE Logistic Regression on baseline data (80% split).
    This model is NEVER retrained — used for concept drift detection.
    """
    n      = len(baseline)
    n_tr   = int(n * TRAIN_SPLIT)
    X_tr   = baseline[FEATURES].iloc[:n_tr].values
    y_tr   = baseline[TARGET].iloc[:n_tr].values
    model  = LogisticRegression(C=10.0, solver='lbfgs', max_iter=500, random_state=42)
    model.fit(X_tr, y_tr)
    return model


def evaluate_stream(model, stream: list) -> list:
    """Evaluate fixed model on each streaming batch. Returns list of accuracies."""
    return [accuracy_score(b[TARGET].values, model.predict(b[FEATURES].values))
            for b in stream]


def save_fig(name: str, fig=None):
    path = os.path.join(PLOT_PATH, f"{name}.png")
    (fig or plt).savefig(path, dpi=150, bbox_inches='tight')
    plt.close('all')
    print(f"  [saved] {path}")


def _add_psi_thresholds(ax, ymax=None):
    """Add PSI threshold reference lines to an axes."""
    ax.axhline(PSI_MOD, color=C_THRESH_MOD, linestyle='--',
               linewidth=1.2, label=f'Moderate drift ({PSI_MOD})', zorder=2)
    ax.axhline(PSI_SIG, color=C_THRESH_SIG, linestyle='--',
               linewidth=1.2, label=f'Significant drift ({PSI_SIG})', zorder=2)
    if ymax:
        ax.set_ylim(0, max(ymax * 1.15, PSI_SIG * 1.3))


def _style_ax(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=8, framealpha=0.8)


print("Utilities loaded.")

Utilities loaded.


## Cell 3 — PSI Computation

Population Stability Index (PSI) is computed in two modes:
- **PSI vs baseline:** compares each streaming batch against the fixed initial training distribution
- **PSI inter-batch:** compares consecutive streaming batches (local stability check)

PSI is averaged across all 5 features (`nswprice`, `nswdemand`, `vicprice`, `vicdemand`, `transfer`).
Thresholds: PSI < 0.10 = no drift, 0.10–0.25 = moderate drift, ≥ 0.25 = significant drift.

In [3]:
def compute_all_psi(baseline: pd.DataFrame, stream: list):
    """
    Returns:
        psi_ref   : list[float] — mean PSI (baseline vs each batch)
        psi_ref_f : list[dict]  — per-feature PSI (baseline vs each batch)
        psi_ib    : list[float] — mean PSI (batch[i-1] vs batch[i])
        psi_ib_f  : list[dict]  — per-feature inter-batch PSI
    """
    psi_ref, psi_ref_f = [], []
    for batch in stream:
        psi_ref.append(compute_psi_mean(baseline, batch))
        psi_ref_f.append(compute_psi_per_feature(baseline, batch))

    psi_ib, psi_ib_f = [np.nan], [{}]   # first batch has no predecessor
    prev = stream[0]
    for batch in stream[1:]:
        psi_ib.append(compute_psi_mean(prev, batch))
        psi_ib_f.append(compute_psi_per_feature(prev, batch))
        prev = batch

    return psi_ref, psi_ref_f, psi_ib, psi_ib_f


print("PSI computation ready.")

PSI computation ready.


## Cell 4 — Plot Functions

Six plot functions are defined here. Each is called once per dataset in the main loop:
- `plot_psi_vs_reference()` — drift magnitude over time relative to the initial distribution
- `plot_psi_interbatch()` — local distribution stability between adjacent batches
- `plot_accuracy_no_retrain()` — predictive degradation of a fixed model (concept drift signal)
- `plot_combined()` — dual-axis overlay to classify each batch as benign/malignant drift
- `plot_feature_importance_drift()` — which features change their predictive contribution over time
- `plot_psi_heatmap()` — per-feature PSI over all batches as a colour-encoded matrix

In [4]:
def plot_psi_vs_reference(psi_ref, stream_ids, dataset_name, label):
    """Plot 1: PSI (baseline vs each batch)."""
    fig, ax = plt.subplots(figsize=(11, 4))
    x = range(len(psi_ref))

    ax.fill_between(x, 0, psi_ref, color=C_BAND, alpha=0.6)
    ax.plot(x, psi_ref, color=C_PSI_REF, linewidth=1.8,
            marker='o', markersize=3, label='PSI (vs baseline)', zorder=3)
    _add_psi_thresholds(ax, max(psi_ref))

    # Shade significant drift zones
    sig_mask = np.array(psi_ref) >= PSI_SIG
    for i, sig in enumerate(sig_mask):
        if sig:
            ax.axvspan(i - 0.5, i + 0.5, color=C_THRESH_SIG, alpha=0.07, zorder=0)

    _style_ax(ax,
              title=f'PSI vs Baseline Reference — {label}',
              xlabel='Streaming Batch Index',
              ylabel='Mean PSI (avg across features)')
    save_fig(f'{dataset_name}_1_psi_vs_reference', fig)


def plot_psi_interbatch(psi_ib, stream_ids, dataset_name, label):
    """Plot 2: PSI between consecutive batches."""
    fig, ax = plt.subplots(figsize=(11, 4))
    x    = range(len(psi_ib))
    vals = np.array(psi_ib, dtype=float)

    ax.fill_between(x, 0, np.nan_to_num(vals), color='#e8fbee', alpha=0.6)
    ax.plot(x, vals, color=C_PSI_IB, linewidth=1.8,
            marker='o', markersize=3, label='PSI (inter-batch)', zorder=3)
    _add_psi_thresholds(ax, float(np.nanmax(vals)))

    _style_ax(ax,
              title=f'PSI Inter-Batch (Consecutive) — {label}',
              xlabel='Streaming Batch Index',
              ylabel='Mean PSI (batch[i-1] → batch[i])')
    save_fig(f'{dataset_name}_2_psi_interbatch', fig)


def plot_accuracy_no_retrain(accuracies, stream_ids, dataset_name, label):
    """Plot 3: Accuracy of fixed model (no retraining)."""
    fig, ax = plt.subplots(figsize=(11, 4))
    x = range(len(accuracies))

    ax.fill_between(x, 0, accuracies, color='#fdecea', alpha=0.5)
    ax.plot(x, accuracies, color=C_ACC, linewidth=1.8,
            marker='o', markersize=3, label='Accuracy (no retrain)', zorder=3)
    ax.axhline(0.80, color='#888', linestyle='--', linewidth=1.1,
               label='Min threshold (0.80)', zorder=2)
    ax.axhline(np.mean(accuracies), color=C_ACC, linestyle=':',
               linewidth=1.0, alpha=0.7,
               label=f'Mean acc = {np.mean(accuracies):.3f}', zorder=2)

    ax.set_ylim(max(0, min(accuracies) - 0.05), 1.02)
    _style_ax(ax,
              title=f'Accuracy Without Retraining — {label}',
              xlabel='Streaming Batch Index',
              ylabel='Batch Accuracy')
    save_fig(f'{dataset_name}_3_accuracy_no_retrain', fig)


def plot_combined(psi_ref, accuracies, stream_ids, dataset_name, label):
    """
    Plot 4: Combined PSI + Accuracy overlay.
    Highlights:
      - High PSI + low accuracy → covariate + concept drift
      - High PSI + stable accuracy → false drift
      - Low PSI + low accuracy → concept drift (no covariate signal)
    """
    fig, ax1 = plt.subplots(figsize=(13, 5))
    x = range(len(psi_ref))

    # Accuracy on primary axis
    ax1.plot(x, accuracies, color=C_ACC, linewidth=2.0,
             label='Accuracy (no retrain)', zorder=4)
    ax1.fill_between(x, 0, accuracies, color=C_ACC, alpha=0.07)
    ax1.axhline(0.80, color='#aaa', linestyle='--', linewidth=0.9,
                label='Min threshold (0.80)')
    ax1.set_ylabel('Batch Accuracy', color=C_ACC, fontsize=10)
    ax1.tick_params(axis='y', labelcolor=C_ACC)
    ax1.set_ylim(max(0, min(accuracies) - 0.08), 1.05)

    # PSI on secondary axis
    ax2 = ax1.twinx()
    ax2.plot(x, psi_ref, color=C_PSI_REF, linewidth=1.8,
             linestyle='-', marker='s', markersize=2.5,
             label='PSI vs baseline', alpha=0.85, zorder=3)
    ax2.axhline(PSI_SIG, color=C_THRESH_SIG, linestyle='--',
                linewidth=1.1, alpha=0.7, label=f'PSI sig. ({PSI_SIG})')
    ax2.set_ylabel('Mean PSI', color=C_PSI_REF, fontsize=10)
    ax2.tick_params(axis='y', labelcolor=C_PSI_REF)
    ax2.set_ylim(0, max(max(psi_ref) * 1.4, PSI_SIG * 1.5))

    # Annotate drift patterns
    psi_arr = np.array(psi_ref)
    acc_arr = np.array(accuracies)
    concept_only = np.where((psi_arr < PSI_SIG) & (acc_arr < 0.80))[0]
    false_drift  = np.where((psi_arr >= PSI_SIG) & (acc_arr >= 0.85))[0]

    for idx in concept_only[:3]:
        ax1.annotate('CD', xy=(idx, acc_arr[idx]),
                     xytext=(idx, acc_arr[idx] - 0.06),
                     fontsize=7, color='#c0392b', ha='center',
                     arrowprops=dict(arrowstyle='->', color='#c0392b', lw=0.8))
    for idx in false_drift[:3]:
        ax2.annotate('FD', xy=(idx, psi_arr[idx]),
                     xytext=(idx, psi_arr[idx] + 0.04),
                     fontsize=7, color='#2980b9', ha='center',
                     arrowprops=dict(arrowstyle='->', color='#2980b9', lw=0.8))

    # Combined legend
    lines1, labs1 = ax1.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labs1 + labs2,
               fontsize=8, loc='lower left', framealpha=0.85, ncol=2)

    ax1.set_xlabel('Streaming Batch Index', fontsize=9)
    ax1.set_title(
        f'Combined Drift View: PSI & Accuracy — {label}\n'
        r'$\bf{CD}$=Concept Drift only  |  $\bf{FD}$=False Drift (PSI↑ Acc stable)',
        fontsize=11)
    ax1.grid(True, alpha=0.2, linestyle='--')
    ax1.spines['top'].set_visible(False)
    save_fig(f'{dataset_name}_4_combined_psi_accuracy', fig)


def plot_feature_importance_drift(baseline: pd.DataFrame,
                                   stream: list,
                                   dataset_name: str, label: str):
    """
    Plot 5: Feature importance comparison — baseline model vs late-batch model.
    Uses logistic regression coefficients as importance proxy.
    """
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline

    n_late = max(1, len(stream) // 4)   # last 25% of stream batches
    late_data = pd.concat(stream[-n_late:]).reset_index(drop=True)

    def fit_lr(data):
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(C=10.0, solver='lbfgs',
                                      max_iter=500, random_state=42))
        ])
        n_tr = int(len(data) * TRAIN_SPLIT)
        pipe.fit(data[FEATURES].iloc[:n_tr], data[TARGET].iloc[:n_tr])
        coefs = np.abs(pipe.named_steps['lr'].coef_[0])
        return coefs / coefs.sum()   # normalised importance

    imp_base = fit_lr(baseline)
    imp_late = fit_lr(late_data)

    x = np.arange(len(FEATURES))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, imp_base, width, label='Baseline model',
                   color='#4878cf', alpha=0.85, edgecolor='white')
    bars2 = ax.bar(x + width/2, imp_late, width, label='Late-batch model',
                   color='#ee854a', alpha=0.85, edgecolor='white')

    # Delta annotations
    for i, (b, l) in enumerate(zip(imp_base, imp_late)):
        delta = l - b
        color = '#c0392b' if abs(delta) > 0.05 else '#555'
        ax.annotate(f'Δ{delta:+.3f}',
                    xy=(i, max(b, l) + 0.01),
                    ha='center', fontsize=8, color=color, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(FEATURES, fontsize=9)
    ax.set_ylabel('Normalised Feature Importance (|coef|)', fontsize=9)
    ax.set_title(
        f'Feature Importance Drift — {label}\n'
        f'Baseline ({int(len(baseline))} rows) vs Late batches (last {n_late} batches)',
        fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    save_fig(f'{dataset_name}_5_feature_importance_drift', fig)


print("Plot functions ready.")


# ============================================================
# CELL 4 — PER-FEATURE PSI HEATMAP
# (bonus: shows WHICH features drive the drift signal)
# ============================================================

def plot_psi_heatmap(psi_ref_f: list, stream_ids,
                      dataset_name: str, label: str):
    """
    Heatmap: PSI per feature over streaming batches.
    Rows = features, Cols = batches.
    """
    data = {f: [d.get(f, 0) for d in psi_ref_f] for f in FEATURES}
    hm_df = pd.DataFrame(data, index=range(len(psi_ref_f))).T

    fig, ax = plt.subplots(figsize=(max(12, len(psi_ref_f) * 0.18), 4))
    sns.heatmap(hm_df, ax=ax, cmap='YlOrRd', vmin=0, vmax=0.5,
                linewidths=0, cbar_kws={'label': 'PSI'})
    ax.set_title(f'Per-Feature PSI Heatmap — {label}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Streaming Batch Index', fontsize=9)
    ax.set_ylabel('Feature', fontsize=9)

    # Mark PSI_SIG contour
    for j in range(hm_df.shape[1]):
        for i, feat in enumerate(FEATURES):
            val = hm_df.iloc[i, j]
            if val >= PSI_SIG:
                ax.add_patch(plt.Rectangle((j, i), 1, 1,
                             fill=False, edgecolor='red', lw=1.2))

    save_fig(f'{dataset_name}_6_psi_heatmap_per_feature', fig)


print("Heatmap function ready.")

Plot functions ready.
Heatmap function ready.


## Cell 5 — Summary Table Builder

Aggregates per-batch PSI and accuracy metrics into a single-row summary per dataset.
Key columns and their interpretation:
- **Mean/Max PSI (ref):** overall drift magnitude vs baseline
- **% Batches PSI ≥ 0.25:** proportion of streaming batches with significant covariate drift
- **Min Accuracy / % Batches Acc < 0.80:** severity of predictive degradation (concept drift)
- **False Drift Batches:** PSI ≥ 0.25 but accuracy ≥ 0.85 — covariate shift without performance impact
- **Missed Drift Batches:** PSI < 0.25 but accuracy < 0.80 — performance degradation without covariate signal

In [5]:
def build_summary_table(results: dict) -> pd.DataFrame:
    """
    One-row-per-dataset summary of drift validation metrics.
    """
    rows = []
    for name, r in results.items():
        psi_arr = np.array(r['psi_ref'])
        acc_arr = np.array(r['accuracies'])
        ib_arr  = np.array(r['psi_ib'])

        rows.append({
            'Dataset'               : DATASET_LABELS[name],
            'Mean PSI (ref)'        : round(float(psi_arr.mean()), 4),
            'Max PSI (ref)'         : round(float(psi_arr.max()), 4),
            '% Batches PSI≥0.25'    : round(float((psi_arr >= PSI_SIG).mean() * 100), 1),
            'Mean PSI (inter-batch)': round(float(np.nanmean(ib_arr)), 4),
            'Mean Accuracy'         : round(float(acc_arr.mean()), 4),
            'Min Accuracy'          : round(float(acc_arr.min()), 4),
            '% Batches Acc<0.80'    : round(float((acc_arr < 0.80).mean() * 100), 1),
            'Concept Drift?'        : '✅' if (acc_arr < 0.80).any() else '—',
            'Covariate Drift?'      : '✅' if (psi_arr >= PSI_SIG).any() else '—',
            'False Drift Batches'   : int(((psi_arr >= PSI_SIG) & (acc_arr >= 0.85)).sum()),
            'Missed Drift Batches'  : int(((psi_arr < PSI_SIG) & (acc_arr < 0.80)).sum()),
        })
    return pd.DataFrame(rows)


print("Summary table function ready.")

Summary table function ready.


## Cell 6 — Main Execution Loop

Iterates over all five datasets. For each dataset:
1. Loads and splits data into baseline and streaming batches
2. Computes PSI vs baseline and PSI inter-batch for all streaming batches
3. Trains a single fixed Logistic Regression on the baseline (never updated)
4. Evaluates the fixed model on each streaming batch (measures concept drift)
5. Generates and saves all 6 plots

Expected output: **30 PNG files** saved to `drift_validation_plots/`.

In [6]:
results = {}

for dataset_name, label in DATASET_LABELS.items():
    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # Load
    print("  Loading dataset...")
    df = load_dataset(dataset_name)
    baseline, stream, stream_ids = get_baseline_stream(df)
    print(f"  Baseline: {len(baseline)} rows | "
          f"Stream: {len(stream)} batches × {BATCH_SIZE} rows")

    # PSI
    print("  Computing PSI...")
    psi_ref, psi_ref_f, psi_ib, psi_ib_f = compute_all_psi(baseline, stream)

    # Fixed model
    print("  Training fixed model (no retraining)...")
    model = train_fixed_model(baseline)
    accuracies = evaluate_stream(model, stream)

    baseline_val_n = int(len(baseline) * TRAIN_SPLIT)
    baseline_val_acc = accuracy_score(
        baseline[TARGET].iloc[baseline_val_n:].values,
        model.predict(baseline[FEATURES].iloc[baseline_val_n:].values)
    )
    print(f"  Baseline validation accuracy: {baseline_val_acc:.4f}")
    print(f"  Stream accuracy  — mean: {np.mean(accuracies):.4f}  "
          f"min: {np.min(accuracies):.4f}")
    print(f"  PSI vs ref       — mean: {np.mean(psi_ref):.4f}  "
          f"max: {np.max(psi_ref):.4f}")

    # Store for summary
    results[dataset_name] = {
        'psi_ref'   : psi_ref,
        'psi_ib'    : psi_ib,
        'accuracies': accuracies,
        'stream_ids': stream_ids,
    }

    # ── Generate all plots ────────────────────────────────────
    print("  Generating plots...")

    plot_psi_vs_reference(psi_ref, stream_ids, dataset_name, label)
    plot_psi_interbatch(psi_ib, stream_ids, dataset_name, label)
    plot_accuracy_no_retrain(accuracies, stream_ids, dataset_name, label)
    plot_combined(psi_ref, accuracies, stream_ids, dataset_name, label)
    plot_feature_importance_drift(baseline, stream, dataset_name, label)
    plot_psi_heatmap(psi_ref_f, stream_ids, dataset_name, label)

    print(f"  ✅ All 6 plots saved for {dataset_name}")

print(f"\n{'='*60}")
print("All datasets processed.")


  Processing: Electricity (Real)
  Loading dataset...
  Baseline: 9000 rows | Stream: 72 batches × 500 rows
  Computing PSI...
  Training fixed model (no retraining)...
  Baseline validation accuracy: 0.7711
  Stream accuracy  — mean: 0.7099  min: 0.5100
  PSI vs ref       — mean: 0.2738  max: 0.4000
  Generating plots...
  [saved] /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/drift_validation_plots/electricity_1_psi_vs_reference.png
  [saved] /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/drift_validation_plots/electricity_2_psi_interbatch.png
  [saved] /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/drift_validation_plots/electricity_3_accuracy_no_retrain.png
  [saved] /content/drive/MyDrive/AM

## Cell 7 — Cross-Dataset Summary & Interpretation

Produces three cross-dataset visualisations and a summary table:
- **Summary table:** one row per dataset with all key drift metrics
- **Multi-panel PSI plot:** side-by-side PSI trends confirming distinct drift signatures
- **Multi-panel accuracy plot:** side-by-side accuracy degradation patterns
- **Drift heatmap:** normalised metric matrix for at-a-glance comparison

**Interpretation guide for summary table:**
- Datasets with `Concept Drift? = ✅` are valid for retraining experiments
- High False Drift Batches confirm that PSI alone is insufficient as a retraining trigger
- The Electricity (Real) dataset shows both drift types with irregular timing,
  making it the most realistic test case
- Gradual/Abrupt covariate-only datasets serve as controls (no concept drift
  expected, accuracy should remain stable despite high PSI)

In [7]:
print("\nGenerating cross-dataset summary plots...")

# 7a. Summary table
summary_df = build_summary_table(results)
print("\n── Drift Validation Summary ──")
print(summary_df.to_string(index=False))

# Save as Excel
summary_path = os.path.join(DATA_PATH, "drift_validation_summary.xlsx")
summary_df.to_excel(summary_path, index=False)
print(f"\nSummary saved → {summary_path}")

# 7b. Three-dataset PSI vs reference comparison (one subplot per dataset)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, (name, r) in zip(axes, results.items()):
    x   = range(len(r['psi_ref']))
    psi = r['psi_ref']
    ax.fill_between(x, 0, psi, color=C_BAND, alpha=0.7)
    ax.plot(x, psi, color=C_PSI_REF, linewidth=1.5)
    ax.axhline(PSI_SIG, color=C_THRESH_SIG, linestyle='--', linewidth=1.0)
    ax.axhline(PSI_MOD, color=C_THRESH_MOD, linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_title(DATASET_LABELS[name].replace(' + ', '\n+'), fontsize=8, fontweight='bold')
    ax.set_xlabel('Batch', fontsize=7)
    ax.set_ylabel('Mean PSI' if ax == axes[0] else '', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('PSI vs Baseline — All Three Datasets', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('summary_psi_all_datasets', fig)

# 7c. Three-dataset accuracy comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (name, r) in zip(axes, results.items()):
    x   = range(len(r['accuracies']))
    acc = r['accuracies']
    ax.fill_between(x, 0, acc, color='#fdecea', alpha=0.5)
    ax.plot(x, acc, color=C_ACC, linewidth=1.5)
    ax.axhline(0.80, color='#888', linestyle='--', linewidth=0.9)
    ax.set_title(DATASET_LABELS[name].replace(' + ', '\n+'), fontsize=8, fontweight='bold')
    ax.set_xlabel('Batch', fontsize=7)
    ax.set_ylabel('Accuracy' if ax == axes[0] else '', fontsize=7)
    ax.set_ylim(0.4, 1.05)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Accuracy Without Retraining — All Three Datasets',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('summary_accuracy_all_datasets', fig)

# 7d. Drift type summary heatmap
fig, ax = plt.subplots(figsize=(10, 4))
hm_data = summary_df[['Mean PSI (ref)', 'Max PSI (ref)',
                        '% Batches PSI≥0.25', 'Mean Accuracy',
                        'Min Accuracy', '% Batches Acc<0.80']].copy()
hm_data.index = [DATASET_LABELS[n] for n in results.keys()]

# Normalise for display
hm_norm = (hm_data - hm_data.min()) / (hm_data.max() - hm_data.min() + 1e-9)
sns.heatmap(hm_norm, annot=hm_data.values, fmt='.2f',
            cmap='RdYlGn_r', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Normalised value'})
ax.set_title('Drift Validation Summary Heatmap\n(raw values annotated, color = normalized)',
             fontsize=11, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
save_fig('summary_drift_heatmap', fig)

print("\n✅ All drift validation plots complete.")
print(f"   Total plots: {len(os.listdir(PLOT_PATH))}")
print(f"   Output path: {PLOT_PATH}")


Generating cross-dataset summary plots...

── Drift Validation Summary ──
                          Dataset  Mean PSI (ref)  Max PSI (ref)  % Batches PSI≥0.25  Mean PSI (inter-batch)  Mean Accuracy  Min Accuracy  % Batches Acc<0.80 Concept Drift? Covariate Drift?  False Drift Batches  Missed Drift Batches
               Electricity (Real)          0.2738         0.4000                54.2                  0.3319         0.7099         0.510                73.6              ✅                ✅                    1                    19
Gradual Covariate + Concept Drift          0.3982         0.9497                61.1                  0.0357         0.9093         0.788                 1.4              ✅                ✅                   33                     0
 Abrupt Covariate + Concept Drift          0.5728         1.0000                66.7                  0.0850         0.8431         0.698                38.9              ✅                ✅                    0                